# Qwen3-VL-4B QLoRA curriculum — seed 1705

Two-stage QLoRA fine-tuning of `Qwen/Qwen3-VL-4B-Instruct`, following the corrected GLM curriculum logic but using a new deterministic seed and validation split. Stage 1 starts from the untouched base model. Stage 2 starts from the best Stage 1 checkpoint, oversamples complex/suspect crops, saves every 0.1 epoch, and retains the best validation candidate.

Training and validation share the same image loader, external area resize, Qwen chat template, prompt, tokenization, greedy decoding, and stopping behavior. The 14 unusable IDs remain excluded. Setup cells do not train; only the two clearly marked stage cells do. Predictions and checkpoints are resumable and remain under `training_outputs/`.


In [ ]:
from pathlib import Path
_requirements = next((path for path in (
    Path.cwd() / 'requirements-qwen-training.txt',
    Path.cwd().parent / 'requirements-qwen-training.txt',
) if path.is_file()), None)
assert _requirements is not None, 'Could not locate requirements-qwen-training.txt.'
%pip install -r {_requirements}


In [ ]:
from __future__ import annotations

import gc, hashlib, inspect, json, math, os, random, re, sys, unicodedata
from dataclasses import asdict
from pathlib import Path

WORKING_DIRECTORY = Path.cwd().resolve()
ROOT = next((path for path in (WORKING_DIRECTORY, *WORKING_DIRECTORY.parents)
             if (path / 'resources/Train.csv').is_file()), None)
assert ROOT is not None, 'Could not locate the project root or resources/Train.csv.'
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))

os.environ['NO_ALBUMENTATIONS_UPDATE'] = '1'
import albumentations, cv2, numpy as np, pandas as pd, peft, torch, transformers
from huggingface_hub import HfApi, snapshot_download
from IPython.display import display
from PIL import Image, ImageOps
from peft import (LoraConfig, PeftModel, TaskType, get_peft_model,
                  prepare_model_for_kbit_training)
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset
from tqdm.auto import tqdm
from transformers import (AutoModelForImageTextToText, AutoProcessor, BitsAndBytesConfig,
                          Trainer, TrainerCallback, TrainingArguments)

from src.barbados_ocr_augmentation import HistoricalHandwritingAugmentation
from src.transcription_metrics import TranscriptionMetrics, ZindiScore

MODEL_ID = 'Qwen/Qwen3-VL-4B-Instruct'
MODEL_DIR = ROOT / 'model/qwen3-vl-4b-instruct'
IMAGES = ROOT / 'resources/images'
NOTEBOOK = ROOT / 'notebooks/qwen3_vl_4b_finetuning.ipynb'
CONFIG = {
    'run_name': 'qwen3_vl_4b_curriculum_seed1705',
    'seed': 1705, 'holdout_size': 400,
    'prompt': ('Transcribe the handwritten text exactly. Preserve original spelling, capitalization, '
               'punctuation, abbreviations, and symbols. Output only the transcription.'),
    'min_spatial_pixels': 65536, 'max_spatial_pixels': 393216,
    'max_sequence_length': 1024, 'max_new_tokens': 128,
    'batch_size': 1, 'gradient_accumulation': 16,
    'lora_rank': 16, 'lora_alpha': 32, 'lora_dropout': 0.05,
    'word_normalizer': 12.0, 'character_normalizer': 55.0,
    'stages': {
        'stage1': {'epochs': 3.0, 'learning_rate': 1e-4, 'warmup_ratio': 0.05,
                   'augmented_views': 4, 'severity': 0.30, 'augmentation_probability': 0.90},
        'stage2': {'epochs': 3.0, 'learning_rate': 3e-5, 'warmup_ratio': 0.03,
                   'augmented_views': 4, 'severity': 0.25, 'augmentation_probability': 0.65,
                   'complex_fraction': 0.40},
    },
}
RUN = ROOT / 'training_outputs' / CONFIG['run_name']
DTYPE = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
assert torch.cuda.is_available(), 'Qwen QLoRA training requires a CUDA GPU.'
assert CONFIG['seed'] != 42, 'This experiment must use a different split from the earlier run.'
print('GPU:', torch.cuda.get_device_name(0), '| run:', RUN, '| seed:', CONFIG['seed'])


## Pin the base model and create the new split

The first run resolves the official model to a commit SHA. The new seed changes the holdout while retaining the same stratification policy and exclusions used by the GLM curriculum.


In [ ]:
def file_hash(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(8 * 1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def object_hash(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False).encode()).hexdigest()

def write_json(path, value):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(value, indent=2, ensure_ascii=False), encoding='utf-8')
    temporary.replace(path)

download_metadata = MODEL_DIR / 'curriculum_download.json'
if download_metadata.is_file() and (MODEL_DIR / 'config.json').is_file():
    model_revision = json.loads(download_metadata.read_text())['revision']
else:
    model_revision = HfApi().model_info(MODEL_ID, revision='main').sha
    snapshot_download(repo_id=MODEL_ID, revision=model_revision, local_dir=MODEL_DIR)
    write_json(download_metadata, {'repo_id': MODEL_ID, 'revision': model_revision})

EXCLUDED_IDS = {
    '79tMUVyfIdy3GzkG','rh8o7bdCGOIBFPwH','F8DYDDp2AvW9Dytw','mfQxfOmeRmwBh0g8',
    'yNyf3Tp0zc7DFj5F','JU7lRwk3jKkus24Z','R6iYPb7MHFiHtXH6','VmrEALeZiP1Y6nF9',
    't0UrASljcgzvBAnO','KH5g92Q3DA5Bo6xi','N78M3v6GKmUzF7sk','WwlTCykxjP3c4kfo',
    'u3b4JNo5bqpfE7Js','MfT9S5oghk9ywNSC',
}
ABBREVIATION = r'\b(?:admrs?|exers?|execs?|pson?s?|prsent?s?|sd|ye|wch|wth|thos|jno|wm|esqr|coll|capt)\b'
ORDINARY_PUNCTUATION = set(".,;:!?\"'()&-/")

def label_flags(text):
    unusual = ''.join(sorted({c for c in text if not
        (c.isalnum() or c.isspace() or c in ORDINARY_PUNCTUATION)}))
    complex_reasons, suspect_reasons = [], []
    if '^' in text: complex_reasons.append('superscript_caret')
    if unusual: complex_reasons.append('unusual_symbols:' + unusual)
    if re.search(ABBREVIATION, text, flags=re.I): complex_reasons.append('abbreviation_pattern')
    if re.search(r'\s{2,}', text): suspect_reasons.append('repeated_whitespace')
    if re.search(r'(?:\bx\b\s*){3,}|_{3,}|\ufffd', text): suspect_reasons.append('placeholder_pattern')
    return '|'.join(complex_reasons), '|'.join(suspect_reasons)

labels = pd.read_csv(ROOT / 'resources/Train.csv', dtype=str, keep_default_na=False)
assert labels.ID.is_unique and EXCLUDED_IDS.issubset(set(labels.ID))
labels = labels.loc[~labels.ID.isin(EXCLUDED_IDS)].reset_index(drop=True)
flags = labels.Target.map(label_flags)
labels['complex_reasons'] = flags.map(lambda value: value[0])
labels['suspect_reasons'] = flags.map(lambda value: value[1])
labels['label_group'] = np.select(
    [labels.suspect_reasons.ne(''), labels.complex_reasons.ne('')],
    ['suspect', 'complex'], default='ordinary')
labels['word_count'] = labels.Target.str.split().str.len()
sizes = []
for image_id in labels.ID:
    with Image.open(IMAGES / f'{image_id}.jpg') as image:
        sizes.append(image.size)
labels['width'] = [size[0] for size in sizes]; labels['height'] = [size[1] for size in sizes]
labels['height_band'] = pd.cut(labels.height, [0, 80, 200, np.inf], labels=['small','medium','large']).astype(str)
labels['length_band'] = pd.cut(labels.word_count, [0, 8, 13, np.inf], labels=['short','medium','long']).astype(str)
strata = labels.label_group + '/' + labels.height_band + '/' + labels.length_band
strata = strata.where(strata.map(strata.value_counts()) >= 12, labels.label_group + '/other')
strata = strata.where(strata.map(strata.value_counts()) >= 2, 'rare')
if strata.value_counts().min() < 2: strata = labels.label_group
train_ids, validation_ids = train_test_split(
    labels.ID, test_size=CONFIG['holdout_size'], random_state=CONFIG['seed'], stratify=strata)
labels['split'] = np.where(labels.ID.isin(validation_ids), 'validation', 'train')
train_df = labels.loc[labels.split.eq('train')].sort_values('ID').reset_index(drop=True)
val_df = labels.loc[labels.split.eq('validation')].sort_values('ID').reset_index(drop=True)
assert len(val_df) == 400 and set(train_df.ID).isdisjoint(val_df.ID)
display(pd.crosstab(labels.label_group, labels.split))


## Lock the run identity

A run can resume only when its model revision, data, split seed, configuration, notebook code, augmentation code, and metric code are unchanged. Use a new `run_name` for any incompatible experiment.


In [ ]:
def notebook_code_hash(path):
    notebook = json.loads(Path(path).read_text(encoding='utf-8'))
    sources = [''.join(cell.get('source', [])) if isinstance(cell.get('source', []), list)
               else str(cell.get('source', '')) for cell in notebook['cells']
               if cell.get('cell_type') == 'code']
    sources = [source for source in sources if not source.lstrip().startswith('%pip install ')]
    return object_hash(sources)

identity = {
    'config': CONFIG, 'model_id': MODEL_ID, 'model_revision': model_revision,
    'train_csv': file_hash(ROOT / 'resources/Train.csv'),
    'excluded_ids': sorted(EXCLUDED_IDS), 'validation_ids': val_df.ID.tolist(),
    'notebook_code': notebook_code_hash(NOTEBOOK),
    'augmentation_code': file_hash(ROOT / 'src/barbados_ocr_augmentation.py'),
    'metrics_code': file_hash(ROOT / 'src/transcription_metrics.py'),
    'versions': {name: module.__version__ for name, module in [
        ('torch', torch), ('transformers', transformers), ('peft', peft),
        ('albumentations', albumentations), ('opencv', cv2), ('numpy', np)]},
}
identity_key = object_hash(identity)
identity_path = RUN / 'run_identity.json'
WARNING_CLEANUP_IDENTITY_KEY = '0894c8a7dd090972926898cbe634c0fd8b91c696a23a5216254918da53be3472'
WARNING_CLEANUP_NOTEBOOK_HASH = 'ecb49716a292433f77e71332bb59acf006348db0bd39cf20c76ef48058352be5'

def compatible_processor_warning_cleanup(saved, current):
    previous = saved.get('identity', {})
    if (saved.get('identity_key') != WARNING_CLEANUP_IDENTITY_KEY or
            previous.get('notebook_code') != WARNING_CLEANUP_NOTEBOOK_HASH):
        return False
    previous_without_code = {key: value for key, value in previous.items() if key != 'notebook_code'}
    current_without_code = {key: value for key, value in current.items() if key != 'notebook_code'}
    return previous_without_code == current_without_code

if identity_path.exists():
    saved = json.loads(identity_path.read_text())
    if saved['identity_key'] != identity_key:
        assert compatible_processor_warning_cleanup(saved, identity), (
            'Run identity changed incompatibly; choose a new run_name.')
        assert not list(RUN.rglob('adapter_model.safetensors')), (
            'Warning-cleanup migration is limited to runs without trained checkpoints.')
        backup = RUN / f"run_identity_before_warning_cleanup_{saved['identity_key'][:16]}.json"
        if not backup.exists():
            write_json(backup, saved)
        write_json(identity_path, {'identity_key': identity_key, 'identity': identity})
        print('Accepted audited processor-warning cleanup; previous identity backed up to', backup)
else:
    RUN.mkdir(parents=True, exist_ok=True)
    assert not any(RUN.iterdir()), 'Nonempty run directory without identity; choose a new run_name.'
    write_json(identity_path, {'identity_key': identity_key, 'identity': identity})
    labels.to_csv(RUN / 'split_manifest.csv', index=False)
print('Run identity:', identity_key)


## Shared preprocessing, curriculum, and Qwen encoding

Augmentation is applied only during training and always before the shared spatial resize. Each occurrence has one clean plus four augmented views. Stage 2 repeats hard rows until complex/suspect exposures are approximately 40%.


In [ ]:
processor = AutoProcessor.from_pretrained(MODEL_DIR, local_files_only=True, trust_remote_code=True)
metric = TranscriptionMetrics(score=ZindiScore(
    word_normalizer=CONFIG['word_normalizer'], character_normalizer=CONFIG['character_normalizer']))

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False; torch.backends.cudnn.deterministic = True

def stable_seed(*parts):
    return int(object_hash([CONFIG['seed'], *parts])[:8], 16)

def normalize(value):
    return unicodedata.normalize('NFC', str(value)).strip()

def load_rgb(image_id):
    with Image.open(IMAGES / f'{image_id}.jpg') as image:
        return ImageOps.exif_transpose(image).convert('RGB')

def resize_spatial(image):
    for bound, comparison in [(CONFIG['max_spatial_pixels'], lambda area, limit: area > limit),
                              (CONFIG['min_spatial_pixels'], lambda area, limit: area < limit)]:
        if comparison(image.width * image.height, bound):
            scale = math.sqrt(bound / (image.width * image.height))
            image = image.resize((max(1, int(image.width * scale)), max(1, int(image.height * scale))),
                                 Image.Resampling.BICUBIC)
    return image

class CurriculumDataset(Dataset):
    def __init__(self, frame, stage):
        self.frame = frame.reset_index(drop=True); self.stage = stage
        settings = CONFIG['stages'][stage]
        self.augmentation = HistoricalHandwritingAugmentation(
            severity=settings['severity'], probability=settings['augmentation_probability'])
        self.set_epoch(0)

    def set_epoch(self, epoch):
        self.epoch = int(epoch); settings = CONFIG['stages'][self.stage]
        rng = np.random.default_rng(stable_seed(self.stage, self.epoch, 'sampling'))
        if self.stage == 'stage1':
            indices = np.arange(len(self.frame))
        else:
            ordinary = np.flatnonzero(self.frame.label_group.eq('ordinary'))
            hard = np.flatnonzero(self.frame.label_group.ne('ordinary'))
            fraction = settings['complex_fraction']
            total = math.ceil(max(len(ordinary) / (1 - fraction), len(hard) / fraction))
            hard_count = max(len(hard), round(total * fraction)); ordinary_count = max(len(ordinary), total-hard_count)
            cover = lambda pool, count: np.concatenate([pool, rng.choice(pool, count-len(pool), replace=True)])
            indices = np.concatenate([cover(ordinary, ordinary_count), cover(hard, hard_count)])
        views = 1 + int(settings['augmented_views'])
        indices = np.repeat(indices, views); variants = np.tile(np.arange(views), len(indices)//views)
        order = rng.permutation(len(indices)); self.indices = indices[order]; self.variants = variants[order]

    def __len__(self): return len(self.indices)

    def __getitem__(self, index):
        row = self.frame.iloc[int(self.indices[index])]; image = load_rgb(row.ID)
        if self.variants[index] > 0:
            image = Image.fromarray(self.augmentation.apply(
                np.asarray(image), seed=stable_seed(self.stage, self.epoch, row.ID, int(index))))
        return {'ID': row.ID, 'image': resize_spatial(image), 'Target': row.Target}

class QwenCollator:
    def __call__(self, examples):
        assert len(examples) == 1
        row = examples[0]
        user = {'role':'user', 'content':[{'type':'image','image':row['image']},
                                           {'type':'text','text':CONFIG['prompt']}]}
        prompt = processor.apply_chat_template(
            [user], tokenize=True, add_generation_prompt=True,
            return_dict=True, return_tensors='pt')
        full = processor.apply_chat_template(
            [user, {'role':'assistant','content':str(row['Target'])}],
            tokenize=True, add_generation_prompt=False, return_dict=True, return_tensors='pt')
        prompt_length = prompt['input_ids'].shape[1]
        assert torch.equal(prompt['input_ids'], full['input_ids'][:, :prompt_length])
        assert full['input_ids'].shape[1] <= CONFIG['max_sequence_length']
        full.pop('token_type_ids', None)
        labels = full['input_ids'].clone(); labels[:, :prompt_length] = -100
        full['labels'] = labels
        return full

collator = QwenCollator()
print('Exposures:', {stage: len(CurriculumDataset(train_df, stage)) for stage in CONFIG['stages']})


## Model loading, validation, and safe checkpoint selection

Qwen is loaded in NF4 4-bit precision and receives rank-16 LoRA on all linear layers, including applicable vision and language projections. Validation uses the complete new 400-image holdout. Stage 1 saves at quarter epochs; Stage 2 saves at each 0.1 epoch. A strict score decrease stops the stage after the checkpoint is safely written.


In [ ]:
BASE_SPEC = {'kind':'base', 'path':None}
CHECKS_PER_EPOCH = {'stage1': 4, 'stage2': 10}

def adapter_path(spec):
    if spec['kind'] == 'base': return MODEL_DIR
    path = RUN / spec['path']; assert (path / 'adapter_model.safetensors').is_file(), path
    return path

def load_model(spec, training=False):
    seed_everything(CONFIG['seed'])
    quantization = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=DTYPE)
    base = AutoModelForImageTextToText.from_pretrained(
        MODEL_DIR, local_files_only=True, trust_remote_code=True, quantization_config=quantization,
        device_map={'':0}, dtype=DTYPE, attn_implementation='sdpa')
    if training:
        base = prepare_model_for_kbit_training(base, use_gradient_checkpointing=True,
                                               gradient_checkpointing_kwargs={'use_reentrant':False})
    if spec['kind'] == 'adapter':
        model = PeftModel.from_pretrained(base, adapter_path(spec), is_trainable=training)
    elif training:
        model = get_peft_model(base, LoraConfig(
            task_type=TaskType.CAUSAL_LM, r=CONFIG['lora_rank'], lora_alpha=CONFIG['lora_alpha'],
            lora_dropout=CONFIG['lora_dropout'], target_modules='all-linear', bias='none'))
    else:
        model = base
    model.config.use_cache = not training
    return model.train() if training else model.eval()

def predict(model, image_id):
    image = resize_spatial(load_rgb(image_id))
    messages = [{'role':'user','content':[{'type':'image','image':image},
                                           {'type':'text','text':CONFIG['prompt']}]}]
    inputs = processor.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_dict=True, return_tensors='pt').to(next(model.parameters()).device)
    inputs.pop('token_type_ids', None); prompt_length = inputs['input_ids'].shape[1]
    old_cache = model.config.use_cache; model.config.use_cache = True
    try:
        with torch.inference_mode():
            output = model.generate(**inputs, max_new_tokens=CONFIG['max_new_tokens'],
                                    num_beams=1, do_sample=False, use_cache=True)
    finally:
        model.config.use_cache = old_cache
    generated = output[0, prompt_length:]
    text = normalize(processor.decode(
        generated, skip_special_tokens=True, clean_up_tokenization_spaces=False))
    return {'Prediction':text, 'generated_tokens':len(generated),
            'hit_token_limit':len(generated) >= CONFIG['max_new_tokens']}

def evaluate(model, frame, stem):
    was_training = model.training; model.eval(); rows = []
    python_state = random.getstate(); numpy_state = np.random.get_state()
    try:
        with torch.random.fork_rng(devices=[0]):
            for row in tqdm(frame.itertuples(index=False), total=len(frame), desc='Validation'):
                rows.append({'ID':row.ID, 'Target':row.Target, **predict(model, row.ID)})
    finally:
        model.train(was_training); random.setstate(python_state); np.random.set_state(numpy_state)
    result = pd.DataFrame(rows); scores = metric(result.Target, result.Prediction)
    scores.update(blank_rate=float(result.Prediction.eq('').mean()),
                  token_limit_rate=float(result.hit_token_limit.mean()),
                  exact_match_rate=float(result.Target.map(normalize).eq(result.Prediction).mean()))
    stem = Path(stem); stem.parent.mkdir(parents=True, exist_ok=True)
    result.to_csv(stem.with_suffix('.csv'), index=False); write_json(stem.with_suffix('.json'), scores)
    return scores

def stage_dir(stage): return RUN / stage
def history(stage):
    path = stage_dir(stage) / 'history.json'
    return json.loads(path.read_text()) if path.exists() else []

def save_history(stage, rows):
    rows = sorted(rows, key=lambda row: row['step']); write_json(stage_dir(stage)/'history.json', rows)
    winner = max(rows, key=lambda row: row['metrics']['zindi_score_proxy'])
    write_json(stage_dir(stage)/'best.json', winner); return winner

def score_decreased(rows):
    return len(rows) >= 2 and rows[-1]['metrics']['zindi_score_proxy'] < rows[-2]['metrics']['zindi_score_proxy']

class StageProgress(TrainerCallback):
    def __init__(self, stage, dataset):
        self.stage, self.dataset = stage, dataset; self.next_epoch = 1/CHECKS_PER_EPOCH[stage]
    def on_train_begin(self, args, state, control, **kwargs):
        checks = CHECKS_PER_EPOCH[self.stage]
        self.next_epoch = (math.floor(float(state.epoch or 0)*checks+1e-9)+1)/checks
    def on_epoch_begin(self, args, state, control, **kwargs):
        self.dataset.set_epoch(int(state.epoch or 0))
    def on_step_end(self, args, state, control, **kwargs):
        epoch = float(state.epoch or 0); checks = CHECKS_PER_EPOCH[self.stage]
        if epoch + 1e-9 >= self.next_epoch or state.global_step >= state.max_steps:
            control.should_save = True; self.next_epoch = (math.floor(epoch*checks+1e-9)+1)/checks
        return control
    def on_save(self, args, state, control, model=None, **kwargs):
        checkpoint = Path(args.output_dir) / f'checkpoint-{state.global_step}'
        metrics = evaluate(model, val_df, stage_dir(self.stage)/'validation'/f'step-{state.global_step}')
        row = {'step':int(state.global_step), 'epoch':float(state.epoch),
               'spec':{'kind':'adapter','path':checkpoint.relative_to(RUN).as_posix()},
               'metrics':metrics}
        rows = [item for item in history(self.stage) if item['step'] != row['step']] + [row]
        save_history(self.stage, rows); print('Saved validation:', row)
        if score_decreased(sorted(rows, key=lambda item:item['step'])):
            control.should_training_stop = True; print('Early stopping after strict score decrease.')
        return control


In [ ]:
def incoming_for(stage):
    if stage == 'stage1': return BASE_SPEC
    best = stage_dir('stage1') / 'best.json'
    assert best.is_file(), 'Finish Stage 1 before starting Stage 2.'
    return json.loads(best.read_text())['spec']

def complete_checkpoints(stage):
    directory = stage_dir(stage) / 'checkpoints'
    if not directory.exists(): return []
    return sorted([path for path in directory.glob('checkpoint-*')
                   if (path/'adapter_model.safetensors').is_file() and (path/'trainer_state.json').is_file()],
                  key=lambda path:int(path.name.split('-')[-1]))

def train_stage(stage):
    directory = stage_dir(stage); directory.mkdir(parents=True, exist_ok=True)
    completed = directory / 'completed.json'
    if completed.exists():
        result = json.loads(completed.read_text()); print('Already complete:', result); return result
    incoming = incoming_for(stage); checkpoints = complete_checkpoints(stage)
    last_checkpoint = checkpoints[-1] if checkpoints else None
    rows = history(stage)
    if not rows:
        baseline = load_model(incoming)
        try: baseline_metrics = evaluate(baseline, val_df, directory/'validation/incoming')
        finally: del baseline; gc.collect(); torch.cuda.empty_cache()
        rows = [{'step':0, 'epoch':0.0, 'spec':incoming, 'metrics':baseline_metrics}]
        save_history(stage, rows)
    if last_checkpoint and int(last_checkpoint.name.split('-')[-1]) not in {row['step'] for row in rows}:
        recovery = load_model({'kind':'adapter','path':last_checkpoint.relative_to(RUN).as_posix()})
        try: recovered = evaluate(recovery, val_df, directory/'validation'/f'step-{last_checkpoint.name.split("-")[-1]}')
        finally: del recovery; gc.collect(); torch.cuda.empty_cache()
        state = json.loads((last_checkpoint/'trainer_state.json').read_text())
        rows.append({'step':int(state['global_step']), 'epoch':float(state['epoch']),
                     'spec':{'kind':'adapter','path':last_checkpoint.relative_to(RUN).as_posix()},
                     'metrics':recovered}); save_history(stage, rows)
    rows = history(stage)
    if score_decreased(rows):
        winner = save_history(stage, rows); write_json(completed, winner); return winner
    dataset = CurriculumDataset(train_df, stage); settings = CONFIG['stages'][stage]
    training_spec = ({'kind':'adapter','path':last_checkpoint.relative_to(RUN).as_posix()}
                     if last_checkpoint else incoming)
    model = load_model(training_spec, training=True)
    args = TrainingArguments(
        output_dir=str(directory/'checkpoints'), per_device_train_batch_size=CONFIG['batch_size'],
        gradient_accumulation_steps=CONFIG['gradient_accumulation'], num_train_epochs=settings['epochs'],
        learning_rate=settings['learning_rate'], warmup_ratio=settings['warmup_ratio'],
        lr_scheduler_type='cosine', weight_decay=0.01, max_grad_norm=1.0,
        bf16=DTYPE==torch.bfloat16, fp16=DTYPE==torch.float16, gradient_checkpointing=True,
        gradient_checkpointing_kwargs={'use_reentrant':False}, save_strategy='no',
        eval_strategy='no', logging_steps=20, report_to='none', remove_unused_columns=False,
        dataloader_num_workers=0, dataloader_pin_memory=False, seed=CONFIG['seed'],
        data_seed=CONFIG['seed'], optim='paged_adamw_8bit')
    trainer = Trainer(model=model, args=args, train_dataset=dataset, data_collator=collator,
                      processing_class=processor, callbacks=[StageProgress(stage, dataset)])
    write_json(directory/'training_plan.json', {'incoming':incoming, 'settings':settings,
        'exposures_per_epoch':len(dataset), 'checkpoint_interval_epochs':1/CHECKS_PER_EPOCH[stage],
        'resume_from':str(last_checkpoint) if last_checkpoint else None})
    try:
        trainer.train(resume_from_checkpoint=str(last_checkpoint) if last_checkpoint else None)
        winner = save_history(stage, history(stage)); write_json(completed, winner)
    finally:
        del trainer, model; gc.collect(); torch.cuda.empty_cache()
    print('Selected:', stage, winner); return winner


## RUN STAGE 1

This long-running cell starts a fresh QLoRA adapter from the pinned Qwen base model and evaluates the new 400-image validation split at quarter-epoch checkpoints.


In [ ]:
stage1_selection = train_stage('stage1')


## RUN STAGE 2

Run only after Stage 1 has produced `best.json`. Stage 2 starts from that best checkpoint, resets the optimizer schedule, oversamples hard crops, saves every 0.1 epoch, and selects its best candidate.


In [ ]:
stage2_selection = train_stage('stage2')
write_json(RUN / 'final_selection.json', stage2_selection)
print('Final selection:', stage2_selection)
